In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mirenahaliti/spoc2data/constellations_udp.py
/kaggle/input/datasets/mirenahaliti/spoc2data/rovers.txt


In [2]:
!grep -n -E "^(import|from)|^class |rovers|    def " /kaggle/input/datasets/mirenahaliti/spoc2data/constellations_udp.py

2:import pykep as pk
3:import numpy as np
4:import scipy
5:import os
6:from matplotlib import pyplot as plt
7:import seaborn as sns
11:from sgp4.api import Satrec, SatrecArray
12:from sgp4.api import WGS72
15:import networkx as nx
61:class constellation_udp:
67:    The constellations must relay information between 7 motherships in orbit and 4 rovers on the surface of New Mars
69:    def __init__(self):
88:        # Latitudes and longitudes of rovers
89:        rovers_db_path = os.path.join(".", "data", "spoc2", "constellations", "rovers.txt")
90:        self.rovers_db = np.loadtxt(rovers_db_path)
91:        self.lambdas = self.rovers_db[:, 0] # latitudes
92:        self.phis = self.rovers_db[:, 1] # longitudes
94:        self.n_rovers = 4
109:    def get_bounds(self):
119:    def get_nix(self):
127:    def get_nobj(self):
135:    def get_nic(self):
143:    def get_rover_constraint(self, lambda0, phi0):
144:        """Evaluate the rover constraint (minimum distance between any two rover

In [3]:
!pip install pykep sgp4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 MB 29.3 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 MB 20.9 MB/s  0:00:04m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 19.0 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pykep]32m4/5 [pykep]]


In [4]:
import os, shutil

src = "/kaggle/input/datasets/mirenahaliti/spoc2data"
os.makedirs("/kaggle/working/data/spoc2/constellations", exist_ok=True)

shutil.copy(src + "/rovers.txt", "/kaggle/working/data/spoc2/constellations/rovers.txt")
shutil.copy(src + "/constellations_udp.py", "/kaggle/working/constellations_udp.py")

os.chdir("/kaggle/working")
print(os.listdir("."))

['.virtual_documents', 'data', 'constellations_udp.py']


In [5]:
from constellations_udp import constellation_udp

udp = constellation_udp()
udp.example()

FileNotFoundError: [Errno 2] No such file or directory: '/usr/local/lib/python3.13/dist-packages/pykep/trajopt/gym/tops/_tops_cr3bp.json'

In [6]:
!grep -n "pk\." /kaggle/input/datasets/mirenahaliti/spoc2data/constellations_udp.py

76:        jd0, fr = pk.epoch(self._t0, 'mjd2000').jd, 0.0 # reference Julian date
81:        self.ep_ref = pk.epoch_from_iso_string("19491231T000000")
98:        self.LOS = 1.05 * pk.EARTH_RADIUS / 1000.0
100:        self.R_p = pk.EARTH_RADIUS / 1000.0
165:        d = scipy.spatial.distance.cdist(pos, pos, lambda u, v: pk.EARTH_RADIUS/1000 * safe_arccos(u, v))
295:        mean_motion = np.sqrt(pk.MU_EARTH/a**3/pk.EARTH_RADIUS**3)
567:        mean_motion = np.sqrt(pk.MU_EARTH / sma**3 / pk.EARTH_RADIUS**3) * 24 * 60 * 60 / (2 * np.pi)
692:        r = pk.EARTH_RADIUS/1000


In [7]:
!mkdir -p /kaggle/working/mm && cd /kaggle/working/mm && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba

bin/micromamba


In [8]:
!/kaggle/working/mm/bin/micromamba create -y -n spoc2 -r /kaggle/working/mm/root -c conda-forge python=3.10 "pykep<3" numpy scipy networkx sgp4 matplotlib seaborn pygmo

[+] 0.0s
[+] 0.1s
conda-forge/linux-64 (check shards) ━━━━━━━╸━━━━━━━━━━━━━   0.0 B Checking  0.1s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.2 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.2 sec)
Fetching and Parsing Packages' Shards                                                    ✔ Done (22.7 sec)

Resolving Environment                                                                     ✔ Done (1.9 sec)

Transaction

  Prefix: /kaggle/working/mm/root/envs/spoc2

  Updating specs:

   - python=3.10
   - pykep<3
   - numpy
   - scipy
   - networkx
   - sgp4
   - matplotlib
   - seaborn
   - pygmo
   - pip


  Package                           Version  Build                  Channel          Size
───────────────────────────────────────────────────────────────────────────────────────────
  Install:
─────────────────────────────────────────────────────────────────────────

In [9]:
!/kaggle/working/mm/root/envs/spoc2/bin/python -c "import pykep; print(pykep.__version__)"

{'major': 2, 'minor': 6}


In [10]:
import os, shutil

src = "/kaggle/input/datasets/mirenahaliti/spoc2data"
os.makedirs("/kaggle/working/data/spoc2/constellations", exist_ok=True)
shutil.copy(src + "/rovers.txt", "/kaggle/working/data/spoc2/constellations/rovers.txt")
shutil.copy(src + "/constellations_udp.py", "/kaggle/working/constellations_udp.py")

code = '''import time
from constellations_udp import constellation_udp

udp = constellation_udp()
t0 = time.time()
udp.example()
print("Koha totale: %.1f sekonda" % (time.time() - t0))
'''
open("/kaggle/working/test1.py", "w").write(code)

print(os.listdir("/kaggle/working"))

['mm', '.virtual_documents', 'test1.py', 'data', 'constellations_udp.py', '__pycache__']


In [11]:
!cd /kaggle/working && MPLBACKEND=Agg /kaggle/working/mm/root/envs/spoc2/bin/python test1.py

Koha totale: 0.0 sekonda
